# Парсинг и анализ данных
## Задание: прочесть файл и подготовить набор инструментов для анализа данных в нём.

In [37]:
#импортируем библиотеки
import re
from typing import Iterator
from functools import wraps

Начнем выполнение работы с декоратора

<i><h3>Этап 7. Метапрограммирование (Декораторы)</h3>

Напишите декоратор @cache_result для кэширования результатов выполнения функции.
Применение: Примените его к самым "тяжелым" аналитическим функциям.

</i>

In [38]:
def cache_result(func):
    cache = {}

    @wraps(func)
    def wrapper(*args, **kwargs):
        # Превращаем args/kwargs в строку (она хэшируема)
        key = (repr(args), repr(sorted(kwargs.items())))
        if key in cache: #если такой ключ уже есть в кэше, возвращаем сохраненный результат
            return cache[key]
        result = func(*args, **kwargs)
        cache[key] = result
        return result

    return wrapper


Этот декоратор применим ниже к функции `find_influential_posts(posts_iterator, /, *, min_likes=1000, min_comments=50)`.

Она анализирует много постов и может вызываться с одинаковыми параметрами.

Также можем применить к функции `create_post_analyzer(filter_func, metric_func)`

<i><h3>Этап 1. Парсинг данных (Строки и Регулярные выражения)</h3>

Напишите функцию `parse_post_block(block: str) -> dict | None`.

Задача: Функция принимает на вход текстовый блок, соответствующий одному посту. Вам необходимо извлечь данные в виде словаря

Итоговый результат: Функция должна вернуть один словарь, содержащий всю информацию о транзакции, включая список товаров.

Требование: Функция должна быть устойчивой к ошибкам и возвращать None для некорректных блоков.</i>

In [39]:
def parse_post_block(block: str) -> dict | None:

    # Основной шаблон поста
    post_pattern = r'''
        POST_ID::\s*(?P<post_id>.+?)\n
        AUTHOR::\s*(?P<author_name>.+?)\s*<(?P<user_id>.+?)>\s*\((?P<email>.+?)\)\n
        TIMESTAMP::\s*(?P<timestamp>.+?)\n
        LIKES::\s*(?P<likes>\d+)\n
        ==BEGIN_TEXT==\n
        (?P<text>.*?)
        \n==END_TEXT==\n
        --COMMENTS--\n
        (?P<comments>.*)
    '''

    match = re.search(post_pattern, block, re.DOTALL | re.VERBOSE)
    if not match: 
        return None

    post_data = match.groupdict()
    
    # Преобразуем лайки
    try:
        post_data['likes'] = int(post_data['likes'])
    except ValueError:
        post_data['likes'] = 0

    # Автор
    post_data['author'] = {
        'username': post_data.pop('author_name').strip(),
        'user_id': post_data.pop('user_id').strip(),
        'email': post_data.pop('email').strip()
    }

    # Текст поста
    post_data['text'] = post_data['text'].strip()

    # Парсим комментарии
    comments_text = post_data.pop('comments')
    post_data['comments'] = []

    comment_pattern = r'\[(?P<comment_user_id>\w+)\]\s*(?P<comment_text>.*?)\s*\(LIKES:\s*(?P<comment_likes>\d+)\)'
    comment_matches = re.finditer(comment_pattern, comments_text, re.DOTALL | re.MULTILINE)

    for comment in comment_matches:
        data = comment.groupdict()
        try:
            likes = int(data['comment_likes'])
        except ValueError:
            likes = 0
        post_data['comments'].append({
            'user_id': data['comment_user_id'],
            'text': data['comment_text'].strip(),
            'likes': likes
        })

    return post_data


In [40]:
block = '''POST_ID:: post-001
AUTHOR:: Alice <user123> (alice@example.com)
TIMESTAMP:: 2025-10-07T11:05:00Z
LIKES:: 152
==BEGIN_TEXT==
This is the first post.
It can span multiple lines.
==END_TEXT==
--COMMENTS--
[user456] This is a great point! (LIKES: 15)
[user789] I agree completely. (LIKES: 4)
[user123] Thanks! (LIKES: 2)
[user888] I have a question about the second line. (LIKES: 7)
_END_POST_
'''
parsed = parse_post_block(block)
print("Результат:")
print(parsed)


Результат:
{'post_id': 'post-001', 'timestamp': '2025-10-07T11:05:00Z', 'likes': 152, 'text': 'This is the first post.\nIt can span multiple lines.', 'author': {'username': 'Alice', 'user_id': 'user123', 'email': 'alice@example.com'}, 'comments': [{'user_id': 'user456', 'text': 'This is a great point!', 'likes': 15}, {'user_id': 'user789', 'text': 'I agree completely.', 'likes': 4}, {'user_id': 'user123', 'text': 'Thanks!', 'likes': 2}, {'user_id': 'user888', 'text': 'I have a question about the second line.', 'likes': 7}]}


<i><h3>Этап 2. Моделирование данных (Классы)</h3>

Создайте класс `Post`.

Задача: Его __init__ принимает словарь, полученный от функции `parse_post_block` и распределяет значения по атрибутам класса (`self.post_id`, `self.likes` и т.д.). Комментарии храните в виде списка словарей, в том же виде, в котором они были сохранены после парсинга. Автора храните в виде словаря в том же виде, в котором он был сохранен после парсинга

Требование: Создайте `@property comment_count`, которое возвращает общее количество комментариев в посте.

</i>

In [41]:
class Post:
    def __init__(self, post_data: dict):
        self.post_id = post_data.get('post_id')
        self.timestamp = post_data.get('timestamp')
        self.likes = post_data.get('likes')
        self.text = post_data.get('text')
        self.author = post_data.get('author', {})  # словарь автора
        self.comments = post_data.get('comments', [])  # список словарей комментариев
        
    @property #свойство
    def comment_count(self) -> int:
        return len(self.comments)
    
    def __str__(self) -> str: #для вывода информации об объекте
        author_name = self.author.get("username")
        return (
            f"Пост: {self.post_id}\n"
            f"Автор: {author_name}\n"
            f"Лайки: {self.likes}\n"
            f"Комментариев: {self.comment_count}\n"
            f"Текст: {self.text[:50]}{'...' if len(self.text) > 50 else ''}"
        )
    

In [42]:
#проверим
print(parsed)
print(Post(parsed))

{'post_id': 'post-001', 'timestamp': '2025-10-07T11:05:00Z', 'likes': 152, 'text': 'This is the first post.\nIt can span multiple lines.', 'author': {'username': 'Alice', 'user_id': 'user123', 'email': 'alice@example.com'}, 'comments': [{'user_id': 'user456', 'text': 'This is a great point!', 'likes': 15}, {'user_id': 'user789', 'text': 'I agree completely.', 'likes': 4}, {'user_id': 'user123', 'text': 'Thanks!', 'likes': 2}, {'user_id': 'user888', 'text': 'I have a question about the second line.', 'likes': 7}]}
Пост: post-001
Автор: Alice
Лайки: 152
Комментариев: 4
Текст: This is the first post.
It can span multiple lines...


<i><h3>Этап 3. Потоковая обработка (Генераторы)</h3>

Напишите генератор `load_posts(file_path: str) -> iter[Post]`.

Задача: Генератор читает файл, накапливает строки до разделителя `_END_POST_`, передает блок в `parse_post_block` и yield-ит экземпляр класса `Post`.

</i>

In [43]:

def load_posts(file_path: str) -> Iterator[Post]:
    current_block = []
    with open(file_path, "r") as file:
        for line in file:
            current_block.append(line)
            if line.strip() == "_END_POST_":
                block = "".join(current_block)
                post_data = parse_post_block(block)
                if post_data:
                    yield Post(post_data)
                current_block = []



In [44]:
#проверим
example_posts = list(load_posts("data/posts.txt")) #сделаем итератор списком для дальнейшего использования
for i, post in enumerate(example_posts):
    if i>=3: break
    print(post, '\n')
    


Пост: post-001
Автор: Alice
Лайки: 152
Комментариев: 3
Текст: This is the first post. It's a great day to start ... 

Пост: post-002
Автор: Bob
Лайки: 890
Комментариев: 2
Текст: Just finished my new project in Python! It was a c... 

Пост: post-003
Автор: Charlie
Лайки: 45
Комментариев: 2
Текст: Does anyone have a good book recommendation? Looki... 



<i><h3>Этап 4. Продвинутые функции (Positional/Keyword-Only & Mutable Defaults) </h3>

Часть А: Напишите функцию со следующей сигнатурой: `group_posts_by_author(post: Post, posts_by_author={})`.

Задача: Эта функция должна группировать посты по их авторам. Она принимает один объект `Post` и словарь. 
Используя ID автора (`post.author['user_id']`) в качестве ключа, функция должна добавить объект post в список постов этого автора в словаре `posts_by_author`. 

Если автора в словаре еще нет, для него нужно создать новый список. `posts_by_author` должен быть опциональным параметром, по умолчанию являющимся пустым словарем. 

Функция должна возвращать измененный словарь `posts_by_author`.

</i>

Если сделать параметром по умолчанию `posts_by_author={}`, обновления словаря не будет с каждым новым вызовом, функция создаст пустой словарь один раз и далее будет использовать его, то есть накапливать авторов. 

Есть второй вариант: Сделаем параметром по умолчанию `posts_by_author=None`. Теперь с каждым вызовом будет создаваться новый словарь (если в функцию не будет передан аргумент `posts_by_author`). Можно реализовать следующим образом:




In [45]:
def group_posts_by_author_var2(post: Post, posts_by_author=None) -> dict:
    if posts_by_author is None:
        posts_by_author = {}
        #остальной код такой же...

Оба варианта имеют смысл, запишем первый вариант, как и требует задание:

In [46]:
def group_posts_by_author(post: Post, posts_by_author={}) -> dict:

    author_id = post.author.get('user_id')
    posts_by_author.setdefault(author_id, []).append(post) #если в словаре нет id, добавляет пустой список
    return posts_by_author


In [47]:
#проверим
grouped = {}
for post in example_posts: #используем тот же список
    grouped = group_posts_by_author(post, grouped)

print("Пример group_posts_by_author:")
for author_id, author_posts in grouped.items():
    print(author_id, ": ", [p.post_id for p in author_posts])



Пример group_posts_by_author:
user123 :  ['post-001', 'post-007', 'post-013', 'post-019', 'post-025', 'post-031', 'post-037']
user456 :  ['post-002', 'post-008', 'post-014', 'post-020', 'post-026', 'post-032', 'post-038']
user789 :  ['post-003', 'post-009', 'post-015', 'post-021', 'post-027', 'post-033', 'post-039']
user101 :  ['post-004', 'post-010', 'post-016', 'post-022', 'post-028', 'post-034', 'post-040']
user202 :  ['post-005', 'post-011', 'post-017', 'post-023', 'post-029', 'post-035', 'post-041']
user303 :  ['post-006', 'post-012', 'post-018', 'post-024', 'post-030', 'post-036', 'post-042']


<i>
Часть Б: Напишите функцию со следующей сигнатурой: `find_influential_posts(posts_iterator, min_likes=1000, min_comments=50)`.

Задача: Функция находит "влиятельные" посты, которые преодолели заданные пороги вовлеченности. Она принимает:

`posts_iterator`: Итератор объектов `Post`. Этот аргумент должен быть `positional-only`.

`min_likes`: Минимальное количество лайков. Этот аргумент должен быть `keyword-only`.

`min_comments`: Минимальное количество комментариев. Этот аргумент должен быть `keyword-only`.

Возвращаемое значение: Список объектов `Post`, которые удовлетворяют обоим условиям (и по лайкам, и по комментариям).

</i>

Для сигнатуры:

Всё до / — только позиционные аргументы.

Всё после * — только именованные (keyword-only) аргументы.

In [48]:
@cache_result #декоратор
def find_influential_posts(posts_iterator, /, *, min_likes=1000, min_comments=50):
    influential_posts = []

    for post in posts_iterator:
        if post.likes >= min_likes and len(post.comments) >= min_comments:
            influential_posts.append(post)

    return influential_posts


In [49]:
#используем
for i, post in enumerate(find_influential_posts(load_posts('data/posts.txt'), min_likes=1000, min_comments=3)):
    print(post, '\n')

Пост: post-010
Автор: Diana
Лайки: 3000
Комментариев: 3
Текст: Big news! Our app just hit 1 million downloads! Th... 



<i><h3>Этап 5. Функциональный подход (Функции высшего порядка) </h3>

Часть А: С помощью `map, filter, sorted` получите:

Итератор постов `PYTHON_POSTS`, содержащих ключевое слово "Python".

Список имен всех авторов `AUTHORS_LIST`, у которых почта зарегистрирована на cooldomain.com.

</i>

In [50]:
posts = load_posts("data/posts.txt")
PYTHON_POSTS = filter(lambda post: "python" in post.text.lower(), posts)
posts_authors = list(load_posts("data/posts.txt")) #делаем итератор списком
AUTHORS_LIST = sorted({post.author["username"] for post in posts_authors if post.author["email"].endswith("@cooldomain.com")}) #множество для удаления дубликатов
AUTHORS_LIST


['Bob', 'Diana', 'Fiona']

<i>
Часть Б: Напишите собственную функцию высшего порядка `create_post_analyzer(filter_func, metric_func)`.

Описание: Эта функция должна принимать два аргумента:

`filter_func`: Функция, которая принимает один объект `Post` и возвращает True или False.

`metric_func`: Функция, которая принимает на вход итератор из отфильтрованных объектов `Post` и вычисляет на их основе одну итоговую метрику (например, среднее число лайков, общее число комментариев).

Возвращаемое значение: `create_post_analyzer` должна возвращать новую функцию. 

Эта новая функция будет принимать на вход один аргумент — итератор всех постов, — и выполнять всю логику: фильтрацию и последующий расчет метрики.

</i>

In [51]:
@cache_result #декоратор
def create_post_analyzer(filter_func, metric_func):
    #создаем функцию
    def analyzer(posts_iterator):
        #применяем фильтрацию
        filtered_posts = list(filter(filter_func, posts_iterator))
        #возвращаем метрику
        return metric_func(filtered_posts)
    #возвращаем функцию
    return analyzer


In [52]:
#проверим
python_filter = lambda post: 'python' in post.text
avg_likes_metric = lambda posts: sum(p.likes for p in posts) / len(list(posts))

python_post_analyzer = create_post_analyzer(python_filter, avg_likes_metric)

result = python_post_analyzer(load_posts("data/posts.txt"))
print("Среднее число лайков у python постов:", result)


Среднее число лайков у python постов: 1195.0


<i><h3>Этап 6. Рекурсивный анализ (Рекурсия) </h3>

Задача: Напишите рекурсивную функцию `parse_quotes(text: str) -> list`.

Логика: В тексте постов (post.text) может встречаться специальная разметка для цитирования: [quote]...[/quote], причем цитаты могут быть вложены друг в друга.

Ваша функция должна разобрать такой текст и вернуть его в виде иерархической структуры.

Пример:
Входной текст: 'Текст до [quote]цитата первого уровня [quote]вложенная цитата[/quote] и ее продолжение[/quote] и текст после.'

Ожидаемый выходной список: ['Текст до ', ['цитата первого уровня ', ['вложенная цитата'], ' и ее продолжение'], ' и текст после.']

</i>

Мы идём по строке слева направо.

Если встречаем [quote], то:

1) запускаем рекурсивный разбор содержимого до соответствующего [/quote];

2) добавляем результат в список как вложенный элемент.

3) Если встречаем [/quote], значит, мы завершили уровень рекурсии — возвращаем текущий список и позицию в тексте.

4) Всё остальное просто добавляем как строки.


In [53]:
def parse_quotes(text: str) -> list:
    def _parse_recursive(pos=0):
        result = []
        current_text = ''
        while pos < len(text):
            if text.startswith('[quote]', pos):
                # добавляем текст до цитаты
                if current_text:
                    result.append(current_text)
                    current_text = ''
                # рекурсивно разбираем вложенный уровень
                nested, pos = _parse_recursive(pos + len('[quote]')) #nested - результат вложенного вызова рекурсивной функции.
                result.append(nested)
            elif text.startswith('[/quote]', pos):
                # добавляем остаток текста и возвращаемся наверх
                if current_text:
                    result.append(current_text)
                return result, pos + len('[/quote]')
            else:
                current_text += text[pos]
                pos += 1
        # если цитата не закрыта просто возвращаем накопленное
        if current_text:
            result.append(current_text)
        return result, pos

    parsed, _ = _parse_recursive(0)
    return parsed


In [54]:
#проверим
text = 'Текст до [quote]цитата первого уровня [quote]вложенная цитата[/quote] и ее продолжение[/quote] и текст после.'
print(parse_quotes(text))


['Текст до ', ['цитата первого уровня ', ['вложенная цитата'], ' и ее продолжение'], ' и текст после.']


<i><h3>Вывод</h3>

При выполнении задания были улучшены навыки в работе с регулярными выражениями, классами, строками, а также изучена работа декораторов, продвинутых функций. Был получен опыт написания рекурсивных функций и работы с файлами.

</i>